# Hands-On: Summarization Bahasa Indonesia dengan mT5 (XL-Sum)

Notebook pendamping **Hands-On Time Sesi 9** (`huggingface.html`) — bagian
*Guided Practical: Summarization*.

Model yang dipakai: **`csebuetnlp/mT5_multilingual_XLSum`** — mT5-base yang
di-fine-tune pada dataset [XL-Sum](https://huggingface.co/datasets/csebuetnlp/xlsum)
(45 bahasa, termasuk Bahasa Indonesia).

## Isi notebook

| Bagian | Isi | Slide |
|---|---|---|
| **0** | Cek lingkungan & muat model (±2,2 GB, sekali unduh) | — |
| **1** | Dua artikel uji (Bahasa Indonesia) | — |
| **2** | Tokenisasi → jumlah token & batas 512 token | 9.3 |
| **3** | Meringkas artikel A & B (inference) | Guided Practical |
| **4** | Efek `max_length` & `num_beams` → kualitas vs waktu | 9.4 |
| **5** | Token → biaya & latency | 9.3 / 9.4 |
| **6** | `pipeline()`: apa yang berubah di transformers 5 | Widget vs Kode |
| **7** | Tugas mandiri | Hands-On |

## Prasyarat

- venv project `rework-transformers` (Python 3.11) dengan `torch`, `transformers`,
  `sentencepiece`, dan `protobuf`. **Perintah pasangnya ada di bagian 0** — tiga
  varian: di dalam notebook (`!pip`), venv + `uv`, atau venv + `pip`.
- Ruang disk ±2,5 GB untuk bobot model — di-cache di
  `~/.cache/huggingface/hub/`, jadi hanya diunduh sekali.
- **CPU sudah cukup.** Di mesin uji: model dimuat ±5 detik, satu ringkasan teks
  pendek ±3 detik.

Jalankan dari atas (**Run All**). Ubah satu angka pun di sel mana saja, lalu jalankan
ulang sel itu — di situlah bagian belajarnya.

> Slide *Guided Practical* di deck memakai `sshleifer/distilbart-cnn-12-6` (khusus
> Inggris). Notebook ini memakai model **multilingual** supaya artikel Bahasa
> Indonesia bisa diringkas apa adanya — sekaligus contoh model *fallback* yang
> sudah diuji mentor.

## 0. Install dependensi & cek lingkungan

Empat paket harus ada:

| Paket | Kenapa |
|---|---|
| `transformers` | `AutoTokenizer`, `AutoModelForSeq2SeqLM`, `generate()` |
| `torch` | mesin komputasi model (sudah ada di venv project & di Colab) |
| `sentencepiece` | tokenizer model ini memakai SentencePiece (`spiece.model`) |
| `protobuf` | dipakai transformers 5 untuk membaca berkas SentencePiece |

Satu perintah, pilih sesuai tempatmu menjalankan notebook:

```bash
# A. Di dalam notebook (Jupyter/Colab) - jalankan sebagai sel:
!pip install -q transformers sentencepiece protobuf

# B. Di terminal project ini (venv + uv, jalur yang dipakai notebook ini):
uv pip install --python .venv/bin/python transformers sentencepiece protobuf

# C. Kalau venv-mu punya pip sendiri:
.venv/bin/python -m pip install transformers sentencepiece protobuf
```

Catatan penting:

- Jangan tambahkan `-U`. Di mesin ini `-U` akan menarik `transformers` 5.17.0 —
  versi yang **belum diuji** untuk sesi ini (yang diuji: 5.16.1). Naikkan versi
  dengan sadar, bukan karena refleks.
- `!pip` di dalam notebook memasang ke kernel yang sedang aktif. Sesudah memasang,
  **restart kernel** lalu jalankan dari atas.
- Kalau `sentencepiece`/`protobuf` belum ada, `AutoTokenizer.from_pretrained()` gagal
  dengan pesan yang menyesatkan (`tiktoken is required`) — bukan soal modelnya.

Sel di bawah mengecek versi keempat paket sekaligus dan menampilkan device yang
dipakai (CPU/GPU). Kalau ada yang `BELUM terpasang`, jalankan perintah di atas lalu
jalankan ulang sel ini.

In [12]:
# ── Pasang dependensi (jalankan sekali saja, hapus tanda # di depannya) ─────────
# Di notebook Jupyter/Colab:
# !pip install -q transformers sentencepiece protobuf
#
# Di terminal project ini (venv + uv):
# uv pip install --python .venv/bin/python transformers sentencepiece protobuf
#
# Kalau venv-mu punya pip:
# .venv/bin/python -m pip install transformers sentencepiece protobuf
#
# Tanpa -U: -U akan menarik transformers 5.17.0 yang belum diuji untuk sesi ini.

import platform
from importlib.metadata import PackageNotFoundError, version

for nama in ["torch", "transformers", "sentencepiece", "protobuf"]:
    try:
        print("{:<14} {}".format(nama, version(nama)))
    except PackageNotFoundError:
        print("{:<14} BELUM terpasang".format(nama))

print("{:<14} {}".format("python", platform.python_version()))

import torch
print("{:<14} {}".format("device", "cuda" if torch.cuda.is_available() else "cpu (CPU)"))

torch          2.14.0
transformers   5.16.1
sentencepiece  0.2.2
protobuf       7.36.1
python         3.11.15
device         cpu (CPU)


## 0.1 Muat model & tokenizer

`from_pretrained()` mengunduh bobot **sekali** lalu menyimpannya di cache. Panggilan
berikutnya memakai cache (hanya beberapa detik).

| Yang perlu diketahui | Nilai |
|---|---|
| Arsitektur | `MT5ForConditionalGeneration` (mT5-base, encoder–decoder) |
| Jumlah parameter | ± 582 juta → ± 2,2 GB bobot fp32 |
| Tokenizer | SentencePiece (`spiece.model`) — butuh `sentencepiece` + `protobuf` |
| Panel API | `pipeline_tag: summarization` |
| Lisensi | **CC BY-NC-SA 4.0 — non-komersial** |

Dua hal terakhir penting untuk keputusan di 9.4:

- **Lisensi.** XL-Sum dan model turunannya berlisensi non-komersial. Untuk produk
  komersial, lisensi ini diskualifikasi — sekeras apa pun hasilnya. Cek kolom
  *lisensi* di Hub sebelum jatuh cinta pada sebuah model.
- **Skor.** Model Card-nya mencantumkan ROUGE Bahasa Indonesia (ROUGE-1 **37,00** /
  ROUGE-2 **17,02** / ROUGE-L **30,76** pada XL-Sum id) — salah satu dari sedikit
  Model Card yang melaporkan skor Indonesia. Tetap rata-rata dataset, bukan jaminan
  untuk teks kamu. Uji dulu, baru putuskan.

In [13]:
import time

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "csebuetnlp/mT5_multilingual_XLSum"

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()                                   # mode inferensi, bukan training
waktu_muat = time.time() - t0

n_param = sum(p.numel() for p in model.parameters()) / 1e6

print("model       :", MODEL_NAME)
print("parameter   : {:.1f} juta".format(n_param))
print("waktu muat  : {:.1f} s".format(waktu_muat))
print("device      :", next(model.parameters()).device)
print("tokenizer   :", type(tokenizer).__name__)
print("vocab       :", tokenizer.vocab_size)
print("token khusus: pad={} eos={} unk={}".format(
    tokenizer.pad_token, tokenizer.eos_token, tokenizer.unk_token))

Loading weights: 100%|██████████| 284/284 [00:00<00:00, 11331.32it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


model       : csebuetnlp/mT5_multilingual_XLSum
parameter   : 582.4 juta
waktu muat  : 3.8 s
device      : cpu
tokenizer   : T5Tokenizer
vocab       : 250100
token khusus: pad=<pad> eos=</s> unk=<unk>


## 1. Dua artikel uji

Teksnya **nyata** — diambil dari Wikipedia Bahasa Indonesia (lisensi teks
**CC BY-SA 4.0**):

- **Artikel A — Kartu Prakerja** (± 3.300 karakter). Sengaja panjang: untuk melihat
  batas **512 token** di bagian 2, dan apa akibatnya pada ringkasan di bagian 3.
- **Artikel B — Kecerdasan buatan** (± 600 karakter). Pendek, muat penuh dalam 512
  token — ringkasan seperti inilah bentuk "kasus ideal" model ini.

Nanti (Tugas 1) ganti dengan artikel pilihanmu sendiri.

In [14]:
ARTIKEL_A = """Kartu Prakerja adalah sebuah program bantuan pelatihan untuk meningkatkan kompetensi angkatan kerja di Indonesia baik dalam konteks keterampilan, pelatihan kembali, dan peningkatan keterampilan. Program ini pertama kali diperkenalkan oleh presiden Joko Widodo pada masa kampanye Pemilihan Umum Presiden Indonesia 2019, bersama dengan Kartu Indonesia Pintar Kuliah dan Kartu sembako murah. Para calon peserta yang memenuhi syarat penerima Kartu Prakerja akan menerima bantuan dana pelatihan sebesar Rp1,000,000 yang kemudian dapat dibelanjakan melalui pelantar digital yang menjadi mitra manajemen pelaksana program. Pelatihan yang dikurasi dari lebih dari 150 lembaga pelatihan dapat dibeli dan diikuti secara daring dari manapun.
== Syarat Pendaftaran ==
WNI berusia 18 tahun ke atas. Tidak sedang menempuh pendidikan formal. Sedang mencari kerja, pekerja/buruh yang terkena PHK, atau pekerja/buruh yang membutuhkan peningkatan kompetensi kerja, seperti pekerja/buruh yang dirumahkan dan pekerja bukan penerima upah, termasuk pelaku usaha mikro & kecil. Bukan penerima bantuan sosial lainnya selama pandemi COVID-19. Bukan Pejabat Negara, Pimpinan dan Anggota DPRD, ASN, Prajurit TNI, Anggota Polri, Kepala Desa dan perangkat desa dan Direksi/Komisaris/Dewan Pengawas pada BUMN atau BUMD. Maksimal 2 NIK dalam 1 KK yang menjadi Penerima Kartu Prakerja.
== Alur Program ==
Daftar melalui lama resmi Kartu Prakerja. Ikuti seleksi dengan tes motivasi dan kemampuan dasar. Daftar gelombang. Tautkan dompet digital untuk penyaluran insentif. Jika lolos gelombang, langsung beli pelatihan. Ikuti pelatihan. Beri penilaian dan ulasan terhadap pelatihan. Dapatkan insentif biaya mencari kerja sebesar Rp600,000/bulan selama 4 bulan.
== Penanganan COVID-19 ==
Selama wabah COVID-19, pemerintah melakukan perubahan terhadap Kartu Prakerja menjadi program semi bantuan sosial untuk membantu para angkatan kerja yang terdampak pemutusan hak kerja (PHK) dan yang baru lulus dari pendidikan formal. Untuk memperluas cakupan bantuan, anggaran dilipatgandakan menjadi Rp20 triliun. Pelatihan yang sebelumnya memiliki 2 skema yakni daring dan langsung diubah menjadi daring secara keseluruhan. Dengan mengeluarkan Kartu Prakerja pada bulan April 2020, maka pemerintah berharap kartu ini akan memberikan manfaat dan solusi seperti: membantu meringankan biaya pelatihan yang ditanggung oleh pihak pekerja dan perusahaan, mengurangi biaya dalam mencari informasi mengenai pelatihan, mendorong keberkerjaan seseorang lewat pengurangan mismatch, dan menjadikan komplemen dari pendidikan formal. Peserta yang mendaftar juga akan mendapatkan uang dengan total Rp3.550.000, terdiri atas biaya pelatihan Rp1.000.000, insentif pascapelatihan Rp600.000 per bulan selama 4 bulan, dan insentif survei keberkerjaan Rp150.000.
== Kontroversi ==
Penunjukan Ruangguru sebagai salah satu mitra program Kartu Prakerja menjadi polemik karena adanya dugaan konflik kepentingan karena CEO Ruangguru, Belva Devara, merupakan Staf Khusus Presiden. Ia pun mundur dari jabatannya sebagai Staf Khusus Presiden.
== Referensi ==
== Pranala luar ==
Situs resmi Kartu Prakerja"""

ARTIKEL_B = """Kecerdasan buatan, juga disebut kecerdasan artifisial dan kepintaran tiruan (bahasa Inggris: artificial intelligence, AI), adalah kecerdasan yang ditunjukkan oleh mesin, yang kontras dengan kecerdasan alami yang ditunjukkan oleh manusia dan hewan. Aplikasi AI yang populer meliputi mesin pencari web canggih (seperti Google Search), sistem rekomendasi (digunakan oleh YouTube, Netflix, dan Amazon), memahami bahasa manusia (seperti Siri dan Alexa), kendaraan otonom (seperti Tesla), alat kreatif (seperti ChatGPT dan Midjourney), serta permainan strategi tingkat tinggi (seperti catur dan Go)."""

print("Artikel A : {:>5} karakter | {} kata".format(len(ARTIKEL_A), len(ARTIKEL_A.split())))
print("Artikel B : {:>5} karakter | {} kata".format(len(ARTIKEL_B), len(ARTIKEL_B.split())))
print()
print("Contoh awal artikel A:")
print(ARTIKEL_A[:180], "...")

Artikel A :  3129 karakter | 418 kata
Artikel B :   593 karakter | 77 kata

Contoh awal artikel A:
Kartu Prakerja adalah sebuah program bantuan pelatihan untuk meningkatkan kompetensi angkatan kerja di Indonesia baik dalam konteks keterampilan, pelatihan kembali, dan peningkatan ...


## 2. Tokenisasi — berapa token yang benar-benar dilihat model?

Ingat 9.3: **jumlah token ≠ jumlah kata**. Untuk mT5, pemetaan teks → token dilakukan
oleh SentencePiece, dan jumlah tokennya jauh lebih besar dari jumlah kata.

Dua hal yang dilakukan `WHITESPACE_HANDLER` (dari Model Card):

1. buang spasi/newline berlebih di tepi teks,
2. rapikan ganti baris dan spasi ganda menjadi satu spasi.

Model ini di-fine-tune pada teks yang sudah dirapikan seperti itu, jadi teks mentah
yang penuh enter bisa menurunkan kualitas ringkasan.

Lalu `max_length=512` + `truncation=True`: teks yang lebih panjang dari 512 token
**dipotong**, bukan diringkas sebagian-sebagian.

In [15]:
import re

WHITESPACE_HANDLER = lambda k: re.sub(r"\s+", " ", re.sub(r"\n+", " ", k.strip()))

MAKS_TOKEN = 512        # batas jendela input model


def token_input(teks, maks=MAKS_TOKEN):
    """Teks mentah -> tensor input_ids (dipangkas ke `maks`, di-pad ke `maks`)."""
    bersih = WHITESPACE_HANDLER(teks)
    return tokenizer(
        bersih,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=maks,
    )["input_ids"]


for nama, teks in [("A - Kartu Prakerja", ARTIKEL_A), ("B - Kecerdasan buatan", ARTIKEL_B)]:
    penuh = tokenizer(WHITESPACE_HANDLER(teks), truncation=False)["input_ids"]
    ids = token_input(teks)
    dipakai = int((ids != tokenizer.pad_token_id).sum())
    print("=" * 72)
    print(nama)
    print("  karakter          : {}".format(len(teks)))
    print("  kata              : {}".format(len(teks.split())))
    print("  token (utuh)      : {}".format(len(penuh)))
    print("  token dikirim     : {}  (maks {})".format(dipakai, MAKS_TOKEN))
    print("  terpotong         : {}".format("ya, {} token".format(len(penuh) - MAKS_TOKEN)
                                              if len(penuh) > MAKS_TOKEN else "tidak"))
    print("  8 token pertama   : {}".format(tokenizer.convert_ids_to_tokens(ids[0][:8])))
    print("  id 8 token itu    : {}".format(ids[0][:8].tolist()))

A - Kartu Prakerja
  karakter          : 3129
  kata              : 418
  token (utuh)      : 783
  token dikirim     : 512  (maks 512)
  terpotong         : ya, 271 token
  8 token pertama   : ['▁Kart', 'u', '▁Pra', 'kerja', '▁adalah', '▁se', 'buah', '▁program']
  id 8 token itu    : [23647, 273, 6222, 39799, 1869, 303, 5601, 2063]
B - Kecerdasan buatan
  karakter          : 593
  kata              : 77
  token (utuh)      : 149
  token dikirim     : 149  (maks 512)
  terpotong         : tidak
  8 token pertama   : ['▁Kec', 'erdas', 'an', '▁buat', 'an', ',', '▁juga', '▁di']
  id 8 token itu    : [37988, 113148, 321, 8722, 321, 261, 1639, 301]


## 3. Meringkas artikel

`model.generate()` adalah inti inference-nya. Parameternya bukan hiasan:

| Parameter | Arti | Efek |
|---|---|---|
| `max_length=84` | panjang maksimum ringkasan (token) | nilai dari Model Card |
| `no_repeat_ngram_size=2` | frasa 2 kata tidak boleh terulang | mencegah model "menggulung" kalimat |
| `num_beams=4` | beam search 4 kandidat | kualitas lebih baik, komputasi 4× lebih berat |
| `torch.no_grad()` | matikan pelacakan gradien | inferensi lebih hemat memori |

Fungsi `ringkas()` di bawah mengembalikan empat hal sekaligus — ringkasan, jumlah
token input, jumlah token output, dan waktu — karena ketiganya yang kita butuhkan di
bagian 5 untuk menghitung biaya dan latency.

In [16]:
def ringkas(teks, maks_ringkasan=84, beam=4):
    """Teks -> (ringkasan, token input, token output, detik)."""
    ids = token_input(teks)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            input_ids=ids,
            max_length=maks_ringkasan,
            no_repeat_ngram_size=2,
            num_beams=beam,
        )
    detik = time.time() - t0

    ringkasan = tokenizer.decode(out[0], skip_special_tokens=True,
                                 clean_up_tokenization_spaces=False)
    n_masuk = int((ids != tokenizer.pad_token_id).sum())
    n_keluar = int((out[0] != tokenizer.pad_token_id).sum())
    return ringkasan, n_masuk, n_keluar, detik

In [17]:
catatan_hasil = []

for nama, teks in [("A - Kartu Prakerja", ARTIKEL_A), ("B - Kecerdasan buatan", ARTIKEL_B)]:
    ringkasan, n_masuk, n_keluar, detik = ringkas(teks)
    catatan_hasil.append({"nama": nama, "n_masuk": n_masuk, "n_keluar": n_keluar, "detik": detik})

    print("=" * 72)
    print("{} | input {} token | output {} token | {:.1f} s".format(
        nama, n_masuk, n_keluar, detik))
    print("=" * 72)
    print(ringkasan)
    print()

A - Kartu Prakerja | input 512 token | output 25 token | 2.4 s
Pendaftaran calon peserta Kartu Prakerja dibuka pada Kamis (09/05) pagi waktu setempat.

B - Kecerdasan buatan | input 149 token | output 39 token | 3.1 s
Pemerintah Indonesia telah mengumumkan bahwa kecerdasan buatan (AI) adalah alat yang kontras dengan mesin, yang ditunjukkan oleh manusia dan hewan.



### Apa yang hilang dari artikel A?

Artikel A lebih panjang dari 512 token, jadi ekornya tidak pernah sampai ke model.
Bandingkan ringkasan artikel A di bagian atas dengan bagian yang terbuang di bawah —
seluruh isi "Penanganan COVID-19" dan "Kontroversi" tidak pernah sampai ke model.

Ini pelajaran praktis dari 9.3: **panjang input adalah keputusan desain.** Kalau
dokumenmu panjang, potong/pilih bagian pentingnya dulu (*pre-processing*), atau
pakai model dengan context window lebih besar — jangan berharap model ajaib.

In [7]:
id_utuh = tokenizer(WHITESPACE_HANDLER(ARTIKEL_A), truncation=False)["input_ids"]
terbuang = id_utuh[MAKS_TOKEN:]

print("token artikel A   : {}".format(len(id_utuh)))
print("dikirim ke model  : {} token".format(MAKS_TOKEN))
print("dibuang           : {} token ({:.0f}% teks)".format(
    len(terbuang), 100 * len(terbuang) / len(id_utuh)))
print()
print("--- bagian yang TIDAK pernah dilihat model ---")
print(tokenizer.decode(terbuang, skip_special_tokens=True, clean_up_tokenization_spaces=False))

token artikel A   : 783
dikirim ke model  : 512 token
dibuang           : 271 token (35% teks)

--- bagian yang TIDAK pernah dilihat model ---
lipatgandakan menjadi Rp20 triliun. Pelatihan yang sebelumnya memiliki 2 skema yakni daring dan langsung diubah menjadi daring secara keseluruhan. Dengan mengeluarkan Kartu Prakerja pada bulan April 2020, maka pemerintah berharap kartu ini akan memberikan manfaat dan solusi seperti: membantu meringankan biaya pelatihan yang ditanggung oleh pihak pekerja dan perusahaan, mengurangi biaya dalam mencari informasi mengenai pelatihan, mendorong keberkerjaan seseorang lewat pengurangan mismatch, dan menjadikan komplemen dari pendidikan formal. Peserta yang mendaftar juga akan mendapatkan uang dengan total Rp3.550.000, terdiri atas biaya pelatihan Rp1.000.000, insentif pascapelatihan Rp600.000 per bulan selama 4 bulan, dan insentif survei keberkerjaan Rp150.000. == Kontroversi == Penunjukan Ruangguru sebagai salah satu mitra program Kartu Prakerja menja

### Baca hasilnya dengan kritis

Kedua ringkasan di atas keluar apa adanya dari model — dan keduanya punya masalah
yang justru layak dibahas:

- Ringkasan artikel A menyebut **"Kamis (09/05)"**. Tanggal itu tidak ada di artikel.
- Ringkasan artikel B menambah **"Pemerintah Indonesia telah mengumumkan"** — juga
  tidak ada di sumber.

Ini *halusinasi*: kalimat yang enak dibaca, tapi menambah fakta yang tidak ada di
teks asli. Bandingkan dengan peringatan di 9.2: skor ROUGE **37,00** di Model Card
itu rata-rata dataset XL-Sum (teks berita), bukan jaminan untuk artikel apa pun.
Karena itu langkah terakhir seleksi model selalu sama: **uji dengan teksmu sendiri.**

Sel di bawah melakukan pemeriksaan kasar: apakah frasa kunci di ringkasan benar-benar
ada di artikel sumbernya.

In [8]:
frasa_kunci = ["Kamis", "09/05", "Pemerintah Indonesia", "diumumkan", "Pendaftaran"]

print("{:<24} {:>16} {:>16}".format("frasa di ringkasan", "ada di artikel A?", "ada di artikel B?"))
print("-" * 58)
for frasa in frasa_kunci:
    print("{:<24} {:>16} {:>16}".format(
        frasa,
        "ya" if frasa in ARTIKEL_A else "TIDAK",
        "ya" if frasa in ARTIKEL_B else "TIDAK"))

print()
print("Baris 'TIDAK' padahal frasanya muncul di ringkasan = indikasi halusinasi.")
print("Cek ini kasar (besar/kecil huruf dihitung) tapi cukup untuk demo.")
print("Di produksi: batasi model pada sumber, atau verifikasi entitas & angka otomatis.")

frasa di ringkasan       ada di artikel A? ada di artikel B?
----------------------------------------------------------
Kamis                               TIDAK            TIDAK
09/05                               TIDAK            TIDAK
Pemerintah Indonesia                TIDAK            TIDAK
diumumkan                           TIDAK            TIDAK
Pendaftaran                            ya            TIDAK

Baris 'TIDAK' padahal frasanya muncul di ringkasan = indikasi halusinasi.
Cek ini kasar (besar/kecil huruf dihitung) tapi cukup untuk demo.
Di produksi: batasi model pada sumber, atau verifikasi entitas & angka otomatis.


## 4. Efek parameter: kualitas vs biaya

Dua tombol yang paling sering diputar orang:

- **`max_length`** — batas panjang ringkasan. Terlalu kecil: ringkasan terpotong di
  tengah kalimat. Terlalu besar: model menambah kalimat yang tidak ada di sumber
  (halusinasi) dan token output bertambah → biaya naik.
- **`num_beams`** — 1 berarti *greedy* (ambil token terbaik tiap langkah). Beam
  lebih besar biasanya lebih baik tapi linier terhadap waktu komputasi.

Perhatikan kolom waktu: itu biaya nyata yang kamu bayar di produksi.

> Angka waktu di sini dari mesin CPU; di GPU angkanya beda, tapi polanya sama.

In [9]:
print("EFEK max_length  (artikel B, num_beams=4)")
print("-" * 72)
for maks in (20, 40, 84):
    r, _, n_keluar, detik = ringkas(ARTIKEL_B, maks_ringkasan=maks)
    print("max_length={:<3} | output {:>2} token | {:>5.1f} s | {}".format(
        maks, n_keluar, detik, r))
    print()

print("EFEK num_beams  (artikel B, max_length=84)")
print("-" * 72)
for beam in (1, 4):
    r, _, n_keluar, detik = ringkas(ARTIKEL_B, maks_ringkasan=84, beam=beam)
    print("num_beams={} | output {:>2} token | {:>5.1f} s | {}".format(
        beam, n_keluar, detik, r))
    print()

EFEK max_length  (artikel B, num_beams=4)
------------------------------------------------------------------------
max_length=20  | output 19 token |   1.6 s | Pemerintah Indonesia telah mengumumkan bahwa kecerdasan buatan (AI) adalah

max_length=40  | output 39 token |   2.9 s | Pemerintah Indonesia telah mengumumkan bahwa kecerdasan buatan (AI) adalah alat yang kontras dengan mesin, yang ditunjukkan oleh manusia dan hewan.

max_length=84  | output 39 token |   3.0 s | Pemerintah Indonesia telah mengumumkan bahwa kecerdasan buatan (AI) adalah alat yang kontras dengan mesin, yang ditunjukkan oleh manusia dan hewan.

EFEK num_beams  (artikel B, max_length=84)
------------------------------------------------------------------------
num_beams=1 | output 50 token |   2.2 s | Dalam waktu dekat, pemerintah Indonesia telah mengumumkan bahwa perusahaan-perusahaan teknologi seperti Google, Facebook, Google dan Google sudah memiliki aplikasi yang digunakan untuk mengatasi kecerdasan buatan.

num

## 5. Token → biaya & latency

Hubungan yang dibangun di 9.3: **token input + token output = biaya**. Angka di bawah
memakai asumsi ilustrasi dari deck ($0,50 per 1.000 token) supaya bisa dihitung di
kepala. Harga API sungguhan beda-beda per penyedia dan per model — dan untuk model
self-hosted, "harga"-nya adalah jam GPU, bukan per token.

In [10]:
HARGA_PER_1K = 0.50          # ilustrasi: $ per 1.000 token (deck 9.3)
VOLUME_BULANAN = 10_000      # ilustrasi: 10.000 ringkasan per bulan


def biaya_usd(n_token_per_ringkasan, volume=VOLUME_BULANAN):
    return n_token_per_ringkasan / 1000 * HARGA_PER_1K * volume


print("{:<24} {:>6} {:>6} {:>10} {:>14}".format(
    "artikel", "in", "out", "per-ringkas", "10k/bulan"))
print("-" * 66)
for h in catatan_hasil:
    total_token = h["n_masuk"] + h["n_keluar"]
    print("{:<24} {:>6} {:>6} {:>10} {:>14}".format(
        h["nama"], h["n_masuk"], h["n_keluar"],
        "${:.5f}".format(total_token / 1000 * HARGA_PER_1K),
        "${:,.2f}".format(biaya_usd(total_token))))

print()
print("Latency terukur (CPU, 4 beam):")
for h in catatan_hasil:
    print("  {:<24} {:>5.1f} s  ->  {:.0f} ms per token output".format(
        h["nama"], h["detik"], 1000 * h["detik"] / max(h["n_keluar"], 1)))
print()
print("Catatan 9.4: model 582 juta parameter ini ringan (CPU pun jalan).")
print("Model besar lewat API menghilangkan urusan VRAM — tapi bayar per token.")

artikel                      in    out per-ringkas      10k/bulan
------------------------------------------------------------------
A - Kartu Prakerja          512     25   $0.26850      $2,685.00
B - Kecerdasan buatan       149     39   $0.09400        $940.00

Latency terukur (CPU, 4 beam):
  A - Kartu Prakerja         2.5 s  ->  102 ms per token output
  B - Kecerdasan buatan      2.8 s  ->  73 ms per token output

Catatan 9.4: model 582 juta parameter ini ringan (CPU pun jalan).
Model besar lewat API menghilangkan urusan VRAM — tapi bayar per token.


## 6. `pipeline()` — dan kenapa di sini tidak dipakai

Model Card model ini ditulis dan diuji pada **transformers 4.11**. Di versi yang
terpasang di lingkungan kita (**5.16.1**), task `summarization` — dan juga
`text2text-generation` — **sudah tidak ada** di daftar task `pipeline()`. Jadi kode
singkat yang biasa dipakai tutorial:

```python
pipeline("summarization", model="csebuetnlp/mT5_multilingual_XLSum")
```

gagal dengan `KeyError: Unknown task summarization`, sementara jalur eksplisit di
bagian 3 (`AutoTokenizer` + `AutoModelForSeq2SeqLM` + `model.generate`) **jalan
normal**. Sel di bawah membuktikannya — bukan mengklaim, tapi menjalankan.

Dua pelajaran yang masuk langsung ke 9.2/9.4:

1. **Contoh kode punya tanggal kadaluarsa.** Model Card ditulis untuk versi library
   tertentu. Cek versinya sebelum menyalahkan modelmu.
2. **`pipeline()` bukan jalan pintas yang selalu ada.** Untuk tugas ini, satu-satunya
   jalur yang didukung versi terpasang adalah API eksplisit — yang justru lebih
   transparan soal parameter (bagus untuk produksi).

In [11]:
import transformers
from transformers import pipeline
from transformers.pipelines import SUPPORTED_TASKS

print("versi transformers        :", transformers.__version__)
print("jumlah task pipeline      :", len(SUPPORTED_TASKS))
print("ada task 'summarization'? :", "summarization" in SUPPORTED_TASKS)
print("ada task 'text2text-generation'?", "text2text-generation" in SUPPORTED_TASKS)

try:
    pipeline("summarization", model=MODEL_NAME)
except KeyError as e:
    print()
    print("pipeline('summarization') -> KeyError:", str(e)[:88], "...")

print()
print("Task yang benar-benar tersedia di versi ini:")
print("  " + ", ".join(sorted(SUPPORTED_TASKS)))

print()
print("Karena itu ringkasan di notebook ini memakai jalur eksplisit:")
print("  AutoTokenizer + AutoModelForSeq2SeqLM + model.generate(...)   (lihat bagian 3)")

versi transformers        : 5.16.1
jumlah task pipeline      : 24
ada task 'summarization'? : False
ada task 'text2text-generation'? False

pipeline('summarization') -> KeyError: "Unknown task summarization, available tasks are ['any-to-any', 'audio-classification',  ...

Task yang benar-benar tersedia di versi ini:
  any-to-any, audio-classification, automatic-speech-recognition, depth-estimation, document-question-answering, feature-extraction, fill-mask, image-classification, image-feature-extraction, image-segmentation, image-text-to-text, keypoint-matching, mask-generation, object-detection, table-question-answering, text-classification, text-generation, text-to-audio, token-classification, video-classification, zero-shot-audio-classification, zero-shot-classification, zero-shot-image-classification, zero-shot-object-detection

Karena itu ringkasan di notebook ini memakai jalur eksplisit:
  AutoTokenizer + AutoModelForSeq2SeqLM + model.generate(...)   (lihat bagian 3)


## 7. Tugas mandiri

Cocokkan dengan slide *Hands-On Time* di deck. Semua tugas di bawah cukup dikerjakan
di notebook ini.

1. **Artikelmu sendiri.** Ganti isi `ARTIKEL_B` dengan artikel Bahasa Indonesia
   pilihanmu (berita, halaman produk, dokumen internal). Jalankan ulang bagian 2 dan 3.
2. **Tebak dulu, baru jalankan.** Sebelum dijalankan, perkirakan: berapa token? apa
   poin penting yang akan tertangkap? Setelah dijalankan, bandingkan tebakanmu dengan
   hasilnya — di mana model melewatkan poin yang menurutmu penting?
3. **Kunci `max_length`.** Cari nilai `max_length` yang ringkasannya paling enak dibaca
   untuk artikelmu (coba 20, 40, 84, 160). Apakah ada titik di mana ringkasan mulai
   menambah informasi yang tidak ada di artikel?
4. **Bandingkan huruf kecil.** Gabungkan daftar *kandidat* hasil eksplorasi Hub kamu
   (tugas ±10 menit di slide sebelumnya) dengan model ini, lalu nilai: mana yang lebih
   enak dibaca, mana yang lebih cepat, mana yang lisensinya boleh dipakai di produk.
5. **Potong dulu atau serahkan apa adanya?** Artikel A kehilangan ± 1/3 isinya karena
   batas 512 token. Rancang cara sederhana untuk mengatasinya di kode (mis. ambil 512
   token pertama + 512 token terakhir, atau ringkas per bagian). Jalankan dan bandingkan
   hasilnya dengan ringkasan di bagian 3.

In [12]:
# ── Tugas 4: uji kandidat model dari eksplorasi Hub ────────────────────────────
# Isi KANDIDAT dengan nama model yang kamu temukan (harus model seq2seq untuk
# summarization, mis. keluarga T5/BART/mT5 — model encoder seperti IndoBERT tidak
# bisa dipakai untuk generate). Kosongkan = sel ini cuma mengingatkan.

KANDIDAT = ""      # contoh: "csebuetnlp/mT5_multilingual_XLSum"

if KANDIDAT:
    tok_k = AutoTokenizer.from_pretrained(KANDIDAT)
    model_k = AutoModelForSeq2SeqLM.from_pretrained(KANDIDAT)

    ids_k = tok_k(WHITESPACE_HANDLER(ARTIKEL_B), return_tensors="pt",
                  truncation=True, max_length=512)["input_ids"]

    t0 = time.time()
    with torch.no_grad():
        out_k = model_k.generate(ids_k, max_length=84, no_repeat_ngram_size=2, num_beams=4)
    print("{:.1f} s -> {}".format(
        time.time() - t0,
        tok_k.decode(out_k[0], skip_special_tokens=True, clean_up_tokenization_spaces=False)))
else:
    print("KANDIDAT masih kosong. Tempel nama model dari Hub, lalu jalankan sel ini lagi.")
    print("Bandingkan hasilnya dengan ringkasan model mT5 di bagian 3.")

KANDIDAT masih kosong. Tempel nama model dari Hub, lalu jalankan sel ini lagi.
Bandingkan hasilnya dengan ringkasan model mT5 di bagian 3.


## Catatan & jebakan yang mudah kena

- **`sentencepiece` + `protobuf` wajib.** Tanpa keduanya, `AutoTokenizer` gagal
  membaca `spiece.model` — errornya menyesatkan (`tiktoken is required`). Ini contoh
  nyata 9.2: Model Card tidak mencantumkan semua dependensi.
- **Batas 512 token itu keras.** Teks lebih panjang dipotong, bukan diringkas
  bertahap. Tentukan strategi (potong, rangkai per bagian, atau model context besar)
  sebelum produksi.
- **`max_length` adalah batas atas, bukan target.** Model berhenti saat menghasilkan
  token `</s>`; output sering lebih pendek dari batasnya.
- **Beam search ≠ gratis.** `num_beams=4` ≈ 4× komputasi. Untuk volume besar, coba
  greedy dulu — kalau kualitasnya cukup, itu penghematan langsung.
- **Lisensi XL-Sum/mT5-XLSum: CC BY-NC-SA 4.0 (non-komersial).** Model bagus tetap
  tidak boleh dipakai di produk komersial kalau lisensinya tidak mengizinkan.
  Cek lisensi **sebelum** uji kualitas.
- **Model ini menyimpan bobot sebagai `pytorch_model.bin`** (bukan safetensors) dan
  tidak punya `tokenizer.json` cepat — pemuatan pertama sedikit lebih lambat. Bukan
  masalah untuk belajar, tapi catat untuk produksi.
- **`pipeline("summarization")` tidak ada di transformers 5.16.1** (dan
  `text2text-generation` juga). Model Card mengacu versi 4.11 — di versi terpasang,
  pakai API eksplisit (bagian 3 & 6). Cek versi sebelum sesi:
  `python -c "import transformers; print(transformers.__version__)"`.
- **Cache ±2,2 GB.** Bersihkan kalau disk sempit: `hf cache list`, lalu
  `hf cache prune`.
- **Selalu uji dengan teksmu sendiri.** ROUGE 37,00 di XL-Sum id adalah rata-rata
  dataset berita; teks teknis/formal bisa jauh berbeda.

## Selanjutnya

1. Ganti `MODEL_NAME` dengan kandidat dari Hub dan ulangi bagian 2–5 — sekaligus
   rasakan bagaimana *model selection* di 9.4 dilakukan dengan angka, bukan kesan.
2. Ukur ROUGE ringkasanmu sendiri memakai `evaluate`/`rouge_score` bila data
   referensinya ada.
3. Bandingkan dengan **API proprietary** (Gemini dari Sesi 1) pada teks yang sama:
   kualitas, waktu, dan hitungan biaya per 1.000 ringkasan.